In [1]:
import sys
import os
from pathlib import Path

ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(ROOT))

print("Añadido al path:", ROOT)

Añadido al path: C:\Users\harra\OneDrive\Documents\GitHub\c2526-R2


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

pd.set_option("display.max_columns", None)

In [3]:
import io
import os
import matplotlib.pyplot as plt

from utils.funciones_minio import crear_cliente_minio, bajar_minio
from utils.config import PATH_PRIMARIOS_LIMPIO

OBJ_VIVIENDAS_VENTA = "viviendas_venta.parquet"
OBJ_VIVIENDAS_ALQUILER = "viviendas_alquiler.parquet"


In [4]:
client = crear_cliente_minio()

In [5]:
df_venta = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_VENTA)
if not isinstance(df_venta, pd.DataFrame):
        df_venta = pd.read_parquet(io.BytesIO(df_venta))
df_alquiler = bajar_minio(client, PATH_PRIMARIOS_LIMPIO, OBJ_VIVIENDAS_ALQUILER)
if not isinstance(df_alquiler, pd.DataFrame):
        df_alquiler = pd.read_parquet(io.BytesIO(df_alquiler))

En este archivo haremos un análisis estadístico sobre el campo Descripción de cada anuncio, así como su relación con los el tipo de Anunciante

In [6]:
df_venta["tipo"] = "venta"
df_alquiler["tipo"] = "alquiler"

df = pd.concat([df_venta, df_alquiler])

Podemos agrupar Agente Pro y Profesional en un solo tipo debido a sus similitudes: Intermediario

In [7]:
def agrupar_tipo(x):
    if x == "Particular":
        return "Particular"
    elif x in ["Agente Pro", "Profesional"]:
        return "Intermediario"
    elif x == "Promotora":
        return "Promotora"

df["grupo"] = df["Anuncia"].apply(agrupar_tipo)

df["grupo"].value_counts()

grupo
Intermediario    27793
Particular        5768
Promotora          737
Name: count, dtype: int64

In [8]:
df["longitud"] = df["Descripcion"].str.len()
df["num_palabras"] = df["Descripcion"].str.split().str.len()

df.groupby("grupo")[["longitud","num_palabras"]].mean().sort_values("longitud")

,longitud,num_palabras
grupo,,
Particular,948.809119,148.324029
Intermediario,1552.393624,244.274386
Promotora,1792.805970,280.278155


Vemos que los anuncios de las promotoras e intermediarios tienden a ser más largos que los de los particulares

In [9]:
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("stopwords")

stopwords_es = stopwords.words("spanish")

vectorizer = CountVectorizer(
    stop_words=stopwords_es,
    max_features=40
)
df = df.reset_index(drop=True)
X = vectorizer.fit_transform(df["Descripcion"].fillna(""))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harra\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [10]:
palabras = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
palabras["grupo"] = df["grupo"]

frecuencias = palabras.groupby("grupo").mean().T
frecuencias.sort_values("Promotora", ascending=False).head(15)

grupo,Intermediario,Particular,Promotora
madrid,1.242255,0.652913,1.937585
zonas,0.709027,0.469140,1.582090
vivienda,1.658619,0.994105,1.246947
dormitorios,0.721441,0.415049,1.078697
zona,1.257295,0.753467,1.006784
espacio,0.608283,0.258495,0.731343
servicios,0.603569,0.343620,0.724559
barrio,0.640665,0.312413,0.698779
ubicación,0.538517,0.316574,0.663501
centro,0.457453,0.344660,0.613297


In [11]:
frecuencias.sort_values("Particular", ascending=False).head(15)

grupo,Intermediario,Particular,Promotora
vivienda,1.658619,0.994105,1.246947
cocina,1.077070,0.991678,0.151967
piso,1.074263,0.935333,0.046133
salón,0.930126,0.807039,0.137042
zona,1.257295,0.753467,1.006784
baño,0.915410,0.694521,0.074627
madrid,1.242255,0.652913,1.937585
dos,0.740006,0.596914,0.337856
metro,0.547872,0.573336,0.442334
cuenta,1.023603,0.506935,0.548168


In [12]:
frecuencias.sort_values("Intermediario", ascending=False).head(15)

grupo,Intermediario,Particular,Promotora
vivienda,1.658619,0.994105,1.246947
zona,1.257295,0.753467,1.006784
madrid,1.242255,0.652913,1.937585
cocina,1.077070,0.991678,0.151967
piso,1.074263,0.935333,0.046133
cuenta,1.023603,0.506935,0.548168
salón,0.930126,0.807039,0.137042
baño,0.915410,0.694521,0.074627
dos,0.740006,0.596914,0.337856
dormitorios,0.721441,0.415049,1.078697


Vectorizamos el texto y comparamos las frecuencias de las palabras. Vemos que algunos terminos claramente se usan más por parte de promotoras e intermediarios, como son "ofrece", "zonas", etc.

In [13]:
df["keyword_empresa"] = df["Descripcion"].str.contains(
    "inmobiliaria|honorarios|gestión|asesor|obra nueva|promoción",
    case=False, regex=True
)

pd.crosstab(df["keyword_empresa"], df["grupo"], normalize="index")

grupo,Intermediario,Particular,Promotora
keyword_empresa,,,
False,0.790926,0.198492,0.010582
True,0.865136,0.082590,0.052274


Aquí vemos determinadas palabras clave que diferencian las intermediarias de los demás anunciantes

In [14]:
df["ratio_mayus"] = df["Descripcion"].apply(
    lambda x: sum(1 for c in x if c.isupper()) / len(x) if isinstance(x,str) else 0
)

df.groupby("grupo")["ratio_mayus"].mean()

grupo
Intermediario    0.042004
Particular       0.041346
Promotora        0.038213
Name: ratio_mayus, dtype: float64

In [ ]:
df["num_exclamaciones"] = df["Descripcion"].str.count("!")


df.groupby("grupo")["num_exclamaciones"].mean()

grupo
Intermediario    0.554060
Particular       0.145978
Promotora        0.294437
Name: num_exclamaciones, dtype: float64

Si bien el análisis de mayúsculas no muestran una diferenciación clara, los intermediarios si que usan muchos más signos de exclamación que las otras.

Tras este análisis podemos observar claramente que existen diferencias significativas entre los tipos diferentes de anunciantes. Por tanto, consideramos que puede ser interesante implementar un modelo que clasifique un anuncio entre estas clases solo conociendo la descripción.